In [1]:
from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.cluster import DBSCAN

from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
processed_dir = Path("../data/processed")
results_dir = Path("../results/clustering")
model_dir = Path("../models")

matrice_tfidf_bigrammes = load_npz(processed_dir / "tfidf_matrix_bigrammes.npz")
model = joblib.load(model_dir / "dbscan_champion.pkl")
labels = pd.read_csv(results_dir / "champion_labels.csv")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
data = pd.read_csv(processed_dir / "sample_clean.csv")


assert matrice_tfidf_bigrammes.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(data["id"])
assert labels["id"].equals(ids["id"])
assert np.array_equal(labels["cluster"].to_numpy(), model.labels_)

In [ ]:
clusters = ids.assign(
    cluster=model.labels_,
    details=data["details"],
    products=data["products"],
    keywords=data["keywords"]
)

In [4]:
pd.set_option("display.max_colwidth", None)
clusters

,id,cluster,details,products
0,1,0,"silky capilene 1 fabric is ultralight, breathable and quick-to-dry exposed, brushed elastic waistband for comfort 5-panel construction with traditional boxer back inseam size m is 4 1/2",Active classic boxers
1,2,-1,"stretch mesh provides support, open-weave mesh for airflow, wicks efficiently, dries fast seamless construction flat-sewn, gusseted inseam is set forward to prevent inner-thigh chafe fly-free support inseam size m is 6",Active sport boxer briefs
2,3,-1,"stretch mesh provides support, open-weave mesh for airflow, wicks efficiently and dries fast seamless construction leg openings have half-inch self-binding for comfort fly-free support",Active sport briefs
3,4,-1,"durable stretch-woven polyester 47% recycled with dwr durable water repellent finish is water- and wind-resistant and highly breathable brushed interior for next-to-skin comfort waistband has belt loops and elastic on back 2-way zippered fly external pockets two front slash, two glued-on thigh, one back hip - all with highly water-resistant, dwr-finished zippers gusseted, zippered cuff with 2-position adjustable settings and tie-down loops",Alpine guide pants
4,5,12,"lightweight, breathable polyester ripstop fabric with deluge dwr durable water repellent finish slightly brushed polyester-mesh liner wicks moisture and dries fast stretch-woven underarm panels provide breathability, stretch for unimpeded range of motion dwr finish on center-front zipper elastic cuffs pockets exterior chest, zippered handwarmers drawcord hem reflective heat-transfer logo",Alpine wind jkt
...,...,...,...,...
495,496,0,"capilene 2 fabric has an open knit that invites airflow and provides excellent wicking for endurance activities in warm to moderate temperatures fastest-drying of our capilene fabrics, it insulates enough for use on chilly days and can be worn beneath other layers elastic waistband brushed for next-to-skin softness side seams set forward for better fit and freedom from chafe functional fly gusseted crotch for unimpeded mobility",Cap 2 bottoms
496,497,0,"capilene 2 fabric has an open knit that invites airflow and provides excellent wicking for endurance activities in warm to moderate temperatures fastest-drying of our capilene fabrics, it is insulates enough for use on chilly days and can be worn beneath other layers self-fabric neckline is soft next to skin raglan sleeves and underarm panels merge out of the way of pack straps flat-rib panel-piecing under the arms and sides is exceptionally breathable, fast drying and eliminates armpit seam chafing",Cap 2 crew
497,498,1,"2.5-layer 100% all-recycled polyester shell with a matte ripstop surface and a waterproof/breathable h2no barrier snap-off detachable storm hood, protective microfleece-lined collar and chin guard 2-way zip laser-cut zippered handwarmer pockets",All-time shell
498,499,4,lightweight organic cotton canvas zip fly with button closure flat front with offset side seams generous wide-entry front pockets welted back pockets hidden under yoke inseam is 12,All-wear cargo shorts


In [ ]:
def item_name_to_id(saisie):
    """
    Cette fonction prend un nom de produit en entrée et renvoie l'identifiant du produit correspondant.
    """
    # Vérifier si le nom du produit est présent dans le DataFrame clusters
    if saisie not in clusters[["products", "keywords"]].values:
        raise ValueError(f"Le nom du produit '{saisie}' n'existe pas dans le DataFrame clusters.")
    
    item_id = clusters.loc[(clusters["products"] == saisie) | (clusters["keywords"] == saisie), "id"].values[0]
    
    return item_id

In [6]:
def recommander(saisie):
    
    while True:
        saisie = saisie.strip()

        try:
            item_id = int(saisie) if saisie.isdigit() else item_name_to_id(saisie)
            input_item = clusters.loc[clusters["id"] == item_id]
        except ValueError as erreur:
            print(erreur)
            saisie = input("Entrez un nom ou un identifiant produit : ")
            continue

        if input_item.empty:
            print("Produit introuvable. Veuillez réessayer.")
        elif input_item["cluster"].iloc[0] == -1:
            print("Ce produit appartient au bruit. Veuillez choisir un autre produit.")
        else:
            break

        saisie = input("Entrez un nom ou un identifiant produit : ")

    cluster = input_item["cluster"].iloc[0]
    cluster_items = clusters.loc[clusters["cluster"] == cluster].copy()

    cluster_items["score"] = cosine_similarity(
        matrice_tfidf_bigrammes[[input_item.index[0]]],
        matrice_tfidf_bigrammes[cluster_items.index.to_numpy()],
    ).ravel()

    cluster_items.sort_values(by="score", ascending=False, inplace=True)
    recommandation = cluster_items.drop(index=input_item.index).head(5)
    
    return recommandation

In [9]:
saisie = input("Entrez un nom ou un identifiant produit : ")
recommander(saisie)

,id,cluster,details,products,score
280,281,12,"light, slightly transparent, tough, triple-ripstop nylon soft shell on torso, arms and shoulders, and breathable recycled polyester stretch-woven fabric on back and underarms jacket has deluge dwr durable water repellent finish for weather protection jacket stuffs into internal chest pocket improved fit on lower sleeve and soft-shell fabric removed - sleeves are easily pushed up over forearms drawcord at hem keeps wind out reflective heat-transfer logos front and back can be worn over baselayers and light midlayers",Nine trails jkt,0.955319
363,364,12,"light, slightly transparent, tough, triple-ripstop nylon soft shell on torso and shoulders breathable recycled polyester stretch-woven fabric on back vest is treated with a deluge dwr durable water repellent finish for weather protection elastic binding at armholes and hem for a good fit center-back reflective zippered pocket reflective heat-transfer logos front and back can be worn over baselayers and light midlayers",Nine trails vest,0.571714
209,210,12,"light, slightly transparent, tough, triple-ripstop nylon soft shell on torso and shoulders breathable recycled polyester stretch-woven fabric on back vest is treated with a deluge dwr durable water repellent finish for weather protection elastic binding at armholes and hem for a good fit reflective zippered pocket on back reflective heat-transfer logos front and back can be worn over baselayers and light midlayers",Nine trails vest,0.571637
4,5,12,"lightweight, breathable polyester ripstop fabric with deluge dwr durable water repellent finish slightly brushed polyester-mesh liner wicks moisture and dries fast stretch-woven underarm panels provide breathability, stretch for unimpeded range of motion dwr finish on center-front zipper elastic cuffs pockets exterior chest, zippered handwarmers drawcord hem reflective heat-transfer logo",Alpine wind jkt,0.276963
307,308,12,"lightweight, breathable polyester ripstop fabric with a deluge dwr durable water repellent finish slightly brushed polyester-mesh liner wicks moisture and dries fast stretch-woven underarm panels provide breathability, stretch for unimpeded range of motion center-front zipper has dwr finish elastic cuffs zippered handwarmer pockets drawcord hem reflective heat-transfer logo",Alpine wind jkt,0.274322
